# Which map is random? Local-attraction replication

This notebook reproduces the 20 mirrored stimuli and the exact model prompt used in the article. The generator is deterministic. No API request runs unless `RUN_MODEL` is set explicitly.

In [ ]:
import base64, io, os, random, re, requests
import matplotlib.pyplot as plt
import pandas as pd

N = 120
ATTRACTION = 0.05
RADIUS = 0.07
STEPS = 12
MASTER_SEED = 7
BASE_TRIALS = 10

QUESTION = (
    'The image shows two squares of dots, labelled LEFT and RIGHT. '
    'Exactly one of them was generated by a truly random process; the other was '
    'generated by a different process. Which square was generated by the truly '
    'random process?\n\nAnswer with exactly one word on the first line: LEFT or '
    'RIGHT. Then one short sentence saying what made you choose it.'
)

In [ ]:
# Exact browser figure: Python port of the component's Mulberry32 generator.
def mulberry32(seed):
    state = seed & 0xffffffff
    def draw():
        nonlocal state
        state = (state + 0x6D2B79F5) & 0xffffffff
        t = ((state ^ (state >> 15)) * (1 | state)) & 0xffffffff
        t = (((t + (((t ^ (t >> 7)) * (61 | t)) & 0xffffffff)) & 0xffffffff) ^ t) & 0xffffffff
        return ((t ^ (t >> 14)) & 0xffffffff) / 4294967296
    return draw

def browser_uniform(seed, n=N):
    rng = mulberry32(seed)
    return [(rng(), rng()) for _ in range(n)]

def article_pair(seed=20260826):
    side_rng = mulberry32(seed)
    random_index = 0 if side_rng() < 0.5 else 1
    uniform = browser_uniform(seed ^ 0x9E3779B9)
    clustered = attract(browser_uniform(seed ^ 0x85EBCA6B))
    return ((uniform, clustered) if random_index == 0 else (clustered, uniform), random_index)

In [ ]:
def uniform_points(rng, n=N):
    return [(rng.random(), rng.random()) for _ in range(n)]

def torus_delta(a, b):
    d = b - a
    if d > 0.5: d -= 1.0
    if d < -0.5: d += 1.0
    return d

def attract(start, strength=ATTRACTION, radius=RADIUS, steps=STEPS):
    points = list(start)
    radius2 = radius * radius
    for _ in range(steps):
        moves = [[0.0, 0.0, 0.0] for _ in points]
        for i in range(len(points)):
            for j in range(i + 1, len(points)):
                dx = torus_delta(points[i][0], points[j][0])
                dy = torus_delta(points[i][1], points[j][1])
                d2 = dx*dx + dy*dy
                if d2 == 0.0 or d2 >= radius2: continue
                weight = 1.0 - d2**0.5 / radius
                moves[i][0] += dx * weight; moves[i][1] += dy * weight; moves[i][2] += weight
                moves[j][0] -= dx * weight; moves[j][1] -= dy * weight; moves[j][2] += weight
        updated = []
        for point, move in zip(points, moves):
            if move[2] == 0.0:
                updated.append(point)
            else:
                updated.append(((point[0] + strength*move[0]/move[2]) % 1.0,
                                (point[1] + strength*move[1]/move[2]) % 1.0))
        points = updated
    return points

def render_pair(left, right):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5.2), dpi=110)
    for ax, points, label in zip(axes, (left, right), ('LEFT', 'RIGHT')):
        ax.scatter([p[0] for p in points], [p[1] for p in points], s=14, c='#1e293b')
        ax.set(xlim=(0, 1), ylim=(0, 1), xticks=[], yticks=[])
        ax.set_title(label, fontsize=13, color='#334155')
        for spine in ax.spines.values(): spine.set_color('#cbd5e1')
    fig.tight_layout()
    return fig

def png_bytes(left, right):
    fig = render_pair(left, right)
    out = io.BytesIO()
    fig.savefig(out, format='png', facecolor='white', metadata={'Software': 'matplotlib'})
    plt.close(fig)
    return out.getvalue()

(article_left, article_right), article_random_index = article_pair()
display(render_pair(article_left, article_right))
print('The article random panel is', ('LEFT', 'RIGHT')[article_random_index])

In [ ]:
master = random.Random(MASTER_SEED)
episodes = []
for index in range(BASE_TRIALS):
    seed = master.randrange(10**9)
    uniform = uniform_points(random.Random(seed))
    clustered = attract(uniform_points(random.Random(seed ^ 0x5DEECE66D)))
    uniform_left = master.random() < 0.5
    left, right = (uniform, clustered) if uniform_left else (clustered, uniform)
    answer = 'LEFT' if uniform_left else 'RIGHT'
    episodes.append({'id': f't{index:02d}', 'left': left, 'right': right, 'answer': answer})
    episodes.append({'id': f't{index:02d}m', 'left': right, 'right': left,
                     'answer': 'RIGHT' if answer == 'LEFT' else 'LEFT'})

display(render_pair(episodes[0]['left'], episodes[0]['right']))
print(QUESTION)
print(f'{len(episodes)} episodes; answer to first episode: {episodes[0]["answer"]}')

## Full opt-in API replication

The cells below run the exact published tasks across all published models (20 mirrored trials × 6 models), save every request and raw response, record provider, tokens, latency and cost, and resume from completed calls.

Paid calls are off by default. Add `OPENROUTER_API_KEY` to Colab Secrets, set `RUN_PAID_EVAL = True`, and choose an aggregate budget. The historical run cost about $0.68; current prices and routing can differ. Each request reserves a conservative worst-case amount before it is sent, so the configured aggregate budget cannot be exceeded.


In [ ]:
from pathlib import Path
import base64, hashlib, json, re, urllib.request

JOBS = [{**episode, "image": png_bytes(episode["left"], episode["right"])} for episode in episodes]
BENCHMARK_ID = "which-map-is-random-clumped-v1"
RUN_PAID_EVAL = False
BUDGET_USD = 5.0
RERUN_DIR = Path("which-map-is-random-clumped-rerun")
MODEL_SPECS = {
    "openai/gpt-5.6-sol": {"input": 2.0, "output": 10.0},
    "google/gemini-3.7-flash": {"input": 0.75, "output": 3.75},
    "anthropic/claude-opus-5": {"input": 5.0, "output": 25.0},
    "anthropic/claude-sonnet-5": {"input": 2.0, "output": 10.0},
    "qwen/qwen3.8-max": {"input": 2.0, "output": 6.0},
    "moonshotai/kimi-k3": {"input": 3.0, "output": 15.0},
    "z-ai/glm-5.3-flash": {"input": 0.075, "output": 0.25},
}
MODELS_TO_RUN = [
    "openai/gpt-5.6-sol", "google/gemini-3.7-flash", "anthropic/claude-opus-5",
    "anthropic/claude-sonnet-5", "qwen/qwen3.8-max", "moonshotai/kimi-k3",
]
RUN_SETTINGS = {"max_tokens": 1600, "reasoning": {"max_tokens": 1024}}

def parse_command(text):
    first = (text or "").strip().upper().splitlines()[0] if (text or "").strip() else ""
    matches = re.findall(r"\b(?:LEFT|RIGHT)\b", first)
    return (matches[0] if len(set(matches)) == 1 else None), first in ("LEFT", "RIGHT")

def build_payload(model, job):
    image = base64.b64encode(job["image"]).decode()
    messages = [{"role": "user", "content": [
        {"type": "text", "text": QUESTION},
        {"type": "image_url", "image_url": {"url": "data:image/png;base64," + image}},
    ]}]
    return openrouter_payload(model, messages, **RUN_SETTINGS)

def job_metadata(job):
    return {"trial_id": job["id"], "image_sha256": hashlib.sha256(job["image"]).hexdigest()}


In [ ]:
import hashlib, os, time, urllib.error
from datetime import datetime, timezone

OPENROUTER_ENDPOINT = "https://openrouter.ai/api/v1/chat/completions"
MAX_NOTEBOOK_BUDGET_USD = 25.0

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def stable_hash(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)
    return hashlib.sha256(encoded.encode()).hexdigest()

def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n")

def append_jsonl(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a") as handle:
        handle.write(json.dumps(value, ensure_ascii=False) + "\n")

def get_api_key():
    key = os.environ.get("OPENROUTER_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("OPENROUTER_API_KEY")
        except Exception:
            key = None
    if not key:
        import getpass
        key = getpass.getpass("OpenRouter API key: ")
    return key

def openrouter_payload(model, messages, *, max_tokens, reasoning, temperature=None):
    spec = MODEL_SPECS[model]
    payload = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "reasoning": reasoning,
        "usage": {"include": True},
        "provider": {
            "sort": "price", "allow_fallbacks": False, "require_parameters": False,
            "max_price": {"prompt": spec["input"], "completion": spec["output"]},
        },
    }
    if temperature is not None:
        payload["temperature"] = temperature
    return payload

def reserve_cost(payload):
    spec = MODEL_SPECS[payload["model"]]
    # UTF-8 bytes safely overestimate prompt tokens, including encoded images.
    prompt_ceiling = len(json.dumps(payload["messages"], ensure_ascii=False).encode()) + 256
    return prompt_ceiling * spec["input"] / 1_000_000 + payload["max_tokens"] * spec["output"] / 1_000_000

def send_openrouter(payload, api_key, timeout=240):
    request = urllib.request.Request(
        OPENROUTER_ENDPOINT,
        data=json.dumps(payload).encode(),
        method="POST",
        headers={
            "Authorization": f"Bearer {api_key}", "Content-Type": "application/json",
            "HTTP-Referer": "https://alestainer.com", "X-OpenRouter-Title": BENCHMARK_ID,
        },
    )
    started = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            status, body = response.status, response.read().decode()
    except urllib.error.HTTPError as error:
        status, body = error.code, error.read().decode(errors="replace")
    try:
        parsed = json.loads(body)
    except json.JSONDecodeError:
        parsed = {"unparsed_body": body}
    return status, parsed, time.perf_counter() - started

def content_of(response):
    try:
        content = response["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError):
        return None
    return content if isinstance(content, str) else None

def usage_of(response):
    usage = response.get("usage") if isinstance(response.get("usage"), dict) else {}
    details = usage.get("completion_tokens_details")
    details = details if isinstance(details, dict) else {}
    cost = usage.get("cost")
    reasoning = details.get("reasoning_tokens")
    return {
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "reasoning_tokens": reasoning if isinstance(reasoning, (int, float)) else None,
        "cost_usd": cost if isinstance(cost, (int, float)) else None,
        "raw_usage": usage,
    }

def result_files():
    return sorted(RERUN_DIR.glob("models/*/calls/*/result.json"))

def load_existing():
    existing = {}
    for path in result_files():
        row = json.loads(path.read_text())
        if row.get("suite_hash") != SUITE_HASH:
            raise RuntimeError(f"Suite mismatch in {path}")
        existing[(row["model"], row["call_id"])] = row
    return existing

def prepare_run():
    if BUDGET_USD <= 0 or BUDGET_USD > MAX_NOTEBOOK_BUDGET_USD:
        raise ValueError(f"BUDGET_USD must be above $0 and at most ${MAX_NOTEBOOK_BUDGET_USD}")
    RERUN_DIR.mkdir(parents=True, exist_ok=True)
    config_path = RERUN_DIR / "config.json"
    config = {
        "created_at": utc_now(), "benchmark_id": BENCHMARK_ID,
        "models": MODELS_TO_RUN, "model_specs": MODEL_SPECS,
        "suite_hash": SUITE_HASH, "run_settings": RUN_SETTINGS,
        "approved_aggregate_budget_usd": BUDGET_USD,
    }
    if config_path.exists():
        prior = json.loads(config_path.read_text())
        for key in ("benchmark_id", "models", "model_specs", "suite_hash", "run_settings"):
            if prior.get(key) != config.get(key):
                raise RuntimeError(f"Resume configuration mismatch: {key}")
        if BUDGET_USD > float(prior["approved_aggregate_budget_usd"]):
            raise RuntimeError("A resume cannot raise the original budget; use a new output directory")
    else:
        write_json(config_path, config)
    existing = load_existing()
    if any(row.get("cost_usd") is None for row in existing.values()):
        raise RuntimeError("Cannot resume safely: a completed call is missing reported cost")
    return get_api_key(), existing, sum(float(row["cost_usd"]) for row in existing.values())

def save_call(*, model, call_id, payload, expected, metadata, existing, api_key, spent):
    key = (model, call_id)
    if key in existing:
        return existing[key], spent, None
    reservation = reserve_cost(payload)
    if spent + reservation > BUDGET_USD + 1e-12:
        return None, spent, f"STOP budget: spent ${spent:.6f}; next call reserves ${reservation:.6f}; cap ${BUDGET_USD:.6f}"
    call_dir = RERUN_DIR / "models" / model.replace("/", "--") / "calls" / call_id
    write_json(call_dir / "request.json", payload)
    status, response, latency = send_openrouter(payload, api_key)
    write_json(call_dir / "raw-response.json", response)
    append_jsonl(RERUN_DIR / "raw-responses.jsonl", {
        "model": model, "call_id": call_id, "http_status": status, "response": response,
    })
    if status < 200 or status >= 300 or "error" in response:
        write_json(call_dir / "error.json", {
            "model": model, "call_id": call_id, "http_status": status,
            "latency_seconds": latency, "response": response, "recorded_at": utc_now(),
        })
        return None, spent, f"STOP API error on {model}/{call_id}: HTTP {status}; rerun to resume"
    raw = content_of(response)
    parsed, strict = parse_command(raw)
    usage = usage_of(response)
    result = {
        "model": model, "returned_model": response.get("model"),
        "provider": response.get("provider"), "call_id": call_id,
        "suite_hash": SUITE_HASH, "expected": expected, "parsed": parsed,
        "strictly_formatted": strict, "correct": parsed == expected,
        "raw_content": raw, "latency_seconds": latency, "recorded_at": utc_now(),
        **metadata, **usage,
    }
    write_json(call_dir / "result.json", result)
    existing[key] = result
    if usage["cost_usd"] is None:
        return result, spent, "STOP accounting: response has no reported cost; rerun to resume"
    spent += float(usage["cost_usd"])
    return result, spent, None


In [ ]:
SUITE_HASH = stable_hash({"benchmark": BENCHMARK_ID, "jobs": [(j["id"], j["answer"], hashlib.sha256(j["image"]).hexdigest()) for j in JOBS], "models": MODELS_TO_RUN, "settings": RUN_SETTINGS})
def run_full_suite():
    api_key, existing, spent = prepare_run()
    stop = None
    total = len(MODELS_TO_RUN) * len(JOBS)
    for order, model in enumerate(MODELS_TO_RUN):
        for job_index, job in enumerate(JOBS):
            call_number = order * len(JOBS) + job_index + 1
            call_id = job["id"]
            if (model, call_id) not in existing:
                print(f"run {call_number:03d}/{total} {model}/{call_id}")
            payload = build_payload(model, job)
            _, spent, stop = save_call(
                model=model, call_id=call_id, payload=payload, expected=job["answer"],
                metadata=job_metadata(job), existing=existing, api_key=api_key, spent=spent,
            )
            if stop:
                print(stop)
                break
        if stop:
            break
    summary = {
        "updated_at": utc_now(), "completed_calls": len(existing), "expected_calls": total,
        "reported_cost_usd": sum(float(r["cost_usd"]) for r in existing.values() if r.get("cost_usd") is not None),
        "correct_calls": sum(r.get("correct") is True for r in existing.values()),
        "missing_cost_calls": sum(r.get("cost_usd") is None for r in existing.values()),
    }
    write_json(RERUN_DIR / "summary.json", summary)
    return summary

print(f"Exact tasks: {len(JOBS)}; selected models: {len(MODELS_TO_RUN)}; planned calls: {len(JOBS) * len(MODELS_TO_RUN)}")
largest_reservation = max(reserve_cost(build_payload(model, job)) for model in MODELS_TO_RUN for job in JOBS)
print(f"Largest conservative single-call reservation: ${largest_reservation:.4f}")
if RUN_PAID_EVAL:
    print(json.dumps(run_full_suite(), indent=2))
else:
    print("DRY RUN ONLY — set RUN_PAID_EVAL = True to send requests")


## Analyze your rerun


In [ ]:
import pandas as pd

rerun_rows = [json.loads(path.read_text()) for path in result_files()]
if not rerun_rows:
    print("No rerun results yet.")
else:
    rerun = pd.DataFrame(rerun_rows)
    display(rerun.groupby("model").agg(calls=("call_id", "size"), correct=("correct", "sum"), cost_usd=("cost_usd", "sum")))


In [ ]:
published = pd.DataFrame([
    ('google/gemini-3.7-flash', 13, 0.025533375),
    ('anthropic/claude-opus-5', 10, 0.27625),
    ('openai/gpt-5.6-sol', 7, 0.04769),
    ('anthropic/claude-sonnet-5', 4, 0.05257),
    ('moonshotai/kimi-k3', 0, 0.2013924),
    ('qwen/qwen3.8-max', 0, 0.076974),
], columns=['model', 'correct_of_20', 'reported_cost_usd'])
published